# Llama-cpp (2026 업데이트판)

llama.cpp로 GGUF 형식의 임베딩 모델을 로컬에서 실행하고, LangChain의 `Embeddings` 인터페이스로 사용하는 방법을 다룹니다.

### 원본 대비 변경 사항
| 항목 | 원본 | 현재 권장 |
|---|---|---|
| 모델 파일 형식 | `ggml-model-q4_0.bin` (GGML) | **GGUF** — GGML `.bin`은 llama.cpp에서 더 이상 로드되지 않음 |
| 모델 준비 | 파일을 직접 받아 경로 지정 | `Llama.from_pretrained()`로 Hugging Face Hub에서 자동 다운로드 |
| 모델 종류 | LLaMA 언어 모델로 임베딩 | **임베딩 전용 모델** 사용 (예: `nomic-embed-text-v1.5`) |
| LangChain 연동 | `langchain_community.embeddings.LlamaCppEmbeddings` | 방법 1: `langchain_core.embeddings.Embeddings`를 상속한 작은 래퍼 클래스<br>방법 2: `llama-server` + `OpenAIEmbeddings` (OpenAI 호환 API) |

**왜 `LlamaCppEmbeddings`를 쓰지 않나요?** 이 클래스는 `langchain-community`에만 있고, 이 패키지는 2026년 5월 지원 종료되었습니다. 전용 통합 패키지가 없는 경우 LangChain 측 권장 방향은 애플리케이션 코드에서 직접 구현하는 것입니다. `Embeddings` 인터페이스는 메서드 두 개(`embed_documents`, `embed_query`)만 구현하면 되므로 어렵지 않고, 인터페이스 구조를 이해하는 좋은 연습이 됩니다.

또한 LLaMA 같은 생성용 LLM의 은닉 상태를 임베딩으로 쓰면 검색 품질이 좋지 않습니다. 임베딩 전용으로 학습된 모델을 사용하세요.

In [ ]:
%pip install -qU llama-cpp-python huggingface_hub langchain-core langchain-openai python-dotenv numpy

> `llama-cpp-python`은 설치 시 C++ 코드를 컴파일합니다. GPU 가속이 필요하면 [설치 문서](https://github.com/abetlen/llama-cpp-python#installation)의 백엔드별 옵션(CUDA, Metal 등)을 참고하세요.

## 환경 설정

- `.env` 파일의 API 키를 `python-dotenv`로 불러옵니다.
- **변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()`는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것이며, 별도 패키지가 필요 없습니다.
- 참고: 임베딩 호출(`embed_query`, `embed_documents`)은 Runnable이 아니어서 LangSmith에 트레이스가 남지 않습니다. 이 챕터에서는 없어도 되는 설정이지만, 이후 체인/에이전트 실습과 형태를 맞추기 위해 둡니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

# LangSmith 추적 (LANGSMITH_API_KEY 는 .env 에 넣어 둡니다)
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH08-Embeddings")

## 방법 1: `llama-cpp-python` + 직접 만든 `Embeddings` 래퍼

### 모델 로드

- `Llama.from_pretrained()`: Hugging Face Hub에서 GGUF 파일을 받아 캐시한 뒤 로드합니다.
- `embedding=True`: 임베딩 모드로 로드합니다.
- 사용 모델: [`nomic-ai/nomic-embed-text-v1.5-GGUF`](https://huggingface.co/nomic-ai/nomic-embed-text-v1.5-GGUF) (Q8_0 양자화, 768차원)

In [ ]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="nomic-ai/nomic-embed-text-v1.5-GGUF",
    filename="nomic-embed-text-v1.5.Q8_0.gguf",
    embedding=True,
    n_ctx=2048,
    verbose=False,
)

# 이미 받아 둔 GGUF 파일이 있다면:
# llm = Llama(model_path="/path/to/model.gguf", embedding=True, verbose=False)

### LangChain `Embeddings` 래퍼 구현

`langchain_core.embeddings.Embeddings`를 상속하고 두 메서드만 구현하면, 벡터 저장소·retriever·`CacheBackedEmbeddings` 등 LangChain의 모든 구성 요소에서 사용할 수 있습니다.

`nomic-embed-text`는 작업 종류를 알려주는 접두어가 필요합니다(쿼리: `search_query: `, 문서: `search_document: `). 래퍼 안에서 자동으로 붙이도록 만들면 사용하는 쪽에서는 신경 쓸 필요가 없습니다.

In [ ]:
from langchain_core.embeddings import Embeddings


class LlamaCppEmbedder(Embeddings):
    """llama-cpp-python 모델을 LangChain Embeddings 인터페이스로 감싼 래퍼."""

    def __init__(self, llm: Llama, query_prefix: str = "", document_prefix: str = ""):
        self.llm = llm
        self.query_prefix = query_prefix
        self.document_prefix = document_prefix

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        inputs = [self.document_prefix + t for t in texts]
        return self.llm.embed(inputs, normalize=True)

    def embed_query(self, text: str) -> list[float]:
        return self.llm.embed(self.query_prefix + text, normalize=True)


llama_embeddings = LlamaCppEmbedder(
    llm,
    query_prefix="search_query: ",
    document_prefix="search_document: ",
)

In [ ]:
text = "This is a test document."  # 테스트용 문서 텍스트

`embed_query(text)`는 텍스트 하나를 벡터로 변환합니다. 검색 시 사용자의 질문을 임베딩할 때 사용합니다.

In [ ]:
query_result = llama_embeddings.embed_query(text)
print(len(query_result))
query_result[:5]

`embed_documents([text])`는 텍스트 리스트를 벡터 리스트로 변환합니다. 검색 대상 문서를 임베딩할 때 사용합니다.

In [ ]:
doc_result = llama_embeddings.embed_documents([text])
print(len(doc_result), len(doc_result[0]))

같은 문장이라도 접두어가 달라 쿼리 벡터와 문서 벡터는 완전히 같지 않습니다. 정규화했으므로 내적이 곧 코사인 유사도입니다.

In [ ]:
import numpy as np

np.dot(query_result, doc_result[0])

## 방법 2: `llama-server` + `OpenAIEmbeddings`

llama.cpp에 포함된 `llama-server`는 OpenAI 호환 `/v1/embeddings` API를 제공합니다. 서버로 띄워 두면 LangChain에서는 `langchain-openai`의 `OpenAIEmbeddings`로 접속할 수 있습니다. 모델을 노트북 프로세스와 분리할 수 있어 여러 앱이 공유하거나 운영 환경으로 옮기기 쉽습니다.

터미널에서 서버를 실행합니다 ([llama.cpp 설치](https://github.com/ggml-org/llama.cpp)).

```bash
llama-server -hf nomic-ai/nomic-embed-text-v1.5-GGUF:Q8_0 --embeddings --pooling mean --port 8080
```

- `check_embedding_ctx_length=False`: 필수 설정입니다. 기본값(`True`)이면 `OpenAIEmbeddings`가 OpenAI 토크나이저(tiktoken)로 텍스트를 토큰 id로 바꿔 보내는데, OpenAI가 아닌 서버는 이를 처리하지 못합니다.
- 서버 방식에서는 접두어를 자동으로 붙여 주지 않으므로 직접 붙입니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings

server_embeddings = OpenAIEmbeddings(
    model="nomic-embed-text-v1.5",       # llama-server는 이름을 검사하지 않음 (기록용)
    base_url="http://localhost:8080/v1",
    api_key="sk-no-key-required",        # 로컬 서버는 키 불필요 (빈 값이면 에러가 나므로 임의 문자열)
    check_embedding_ctx_length=False,
)

server_query = server_embeddings.embed_query("search_query: " + text)
len(server_query)